# Lab 3: Evaluate and score the Northwind HR assistant

You will grade the assistant's answers four ways and see each grade appear in Langfuse:

| Part | What you do | Claude cost |
|---|---|---|
| A | Score traces you already recorded, with the test suite's own checks | none |
| B | Turn the 22 test questions into a Langfuse **dataset** | none |
| C | Run the dataset as an **experiment**, graded by checks and **LLM judges** | about $0.07 |
| D | Write your own evaluator | about $0.01 |
| E | Grade answers yourself, then compare with the judges | none |
| F | Make a change that looks harmless, and catch the **regression** | about $0.04 |

The explanations are in **docs/langfuse-lab/03-evaluation-scoring.md**. Read each part there first,
then run its cells here, top to bottom. Every cell uses the real assistant: nothing here copies its logic.

## Setup

In [ ]:
import warnings
import pandas as pd
from IPython.display import display

# Experiments show a progress bar; this hides a harmless warning about its notebook display
warnings.filterwarnings("ignore", message="IProgress not found")

from app import config, evaluate, experiment, judges, tracing

pd.set_option("display.max_colwidth", 120)
print("tracing on:", tracing.ENABLED, "| keys accepted:", tracing.langfuse.auth_check())
print("assistant model:", config.CLAUDE_MODEL, "| judge model:", judges.JUDGE_MODEL)

Both must say `True`. If not, see Part 2 of the lab, "If something does not show up".

The score definitions (what `golden_check`, `faithfulness` and the others mean, and their type) are
created once. Running this again changes nothing.

In [ ]:
config_ids = experiment.ensure_score_configs()
pd.DataFrame([{"score": c["name"], "type": c["data_type"], "meaning": c["description"]}
              for c in experiment.SCORE_CONFIGS])

## Part A: Score traces you already have

Each time `python -m app.evaluate` ran, it recorded a trace per question, tagged with the run's id in
`eval_run`. The trace stores the answer, so it can be graded again later without asking Claude.

Find the runs:

In [ ]:
roots = tracing.langfuse.api.observations.get_many(is_root_observation=True, fields="core,basic,metadata", limit=1000).data
runs = sorted({(r.metadata or {}).get("eval_run") for r in roots} - {None})
print("evaluate runs found:", runs)

Grade the latest run with `evaluate.check`, the same function `python -m app.evaluate` uses, and attach the
result to each trace as a `golden_check` score. The score's comment lists anything that failed.

In [ ]:
latest_run = runs[-1]
results = experiment.score_existing_traces(latest_run)
print(f"{sum(results.values())} of {len(results)} traces passed in run {latest_run}")

**Look in Langfuse:** open **Tracing**, open any trace tagged `evaluation`, and find the `golden_check` score.
Then open **Scores** in the sidebar: every score you create is listed there.

## Part B: The dataset

A **dataset** is a fixed list of inputs with what a good answer must do. Running the assistant over the
same dataset after every change shows whether the change helped.

In [ ]:
print("items:", experiment.sync_dataset())
items = tracing.langfuse.get_dataset(experiment.DATASET).items
pd.DataFrame([{"case": i.id, "trap": i.metadata["trap"], "question": i.input["question"],
               "follows": " / ".join(i.input["after"])} for i in items])

`data/evaluation.json` stays the one place test cases are written. Each item's id is the case id, so
syncing again updates the items rather than adding new ones.

**Look in Langfuse:** **Datasets** → `northwind-golden`. Open an item to see its input and expected output.

### The judges that run inside Langfuse

Three judges grade every answer Claude writes, on the `claude-answer` observation: **faithfulness**,
**citation_accuracy** and **relevance**. This creates them, and the rule that runs them, if they are missing.

In [ ]:
experiment.setup_managed_judges()

**Look in Langfuse:** **Evaluators**. Open `faithfulness` to read its prompt and its variable mapping.

## Part C: A baseline experiment

An **experiment** runs every dataset item through the assistant and grades each answer:

- `golden_check`: `evaluate.check`, unchanged, as an evaluator
- `correctness`: an LLM judge that compares the answer with the item's expectations
- the three Langfuse judges, which score each answer a few seconds after it arrives

This is the code that turns `evaluate.check` into an evaluator. It is all there is to it:

In [ ]:
import inspect
print(inspect.getsource(experiment.golden_check))

Run the baseline. It takes about a minute and a half and costs about $0.07.

In [ ]:
baseline = experiment.run("baseline", judge=True)
baseline_rows = experiment.summarise(baseline)

Collect every score of the run. This waits until the Langfuse judges have finished, usually 10 to 30 seconds.

In [ ]:
baseline_scores = pd.DataFrame.from_dict(experiment.experiment_scores("baseline"), orient="index")
baseline_scores.drop(columns="trace_id").mean().round(2)

Where did a judge disagree with the check? `golden_check` only looks for words; the judges read for meaning.

In [ ]:
cols = ["golden_check", "correctness", "faithfulness", "citation_accuracy", "relevance"]
baseline_scores[(baseline_scores[cols] < 1).any(axis=1)][cols]

Read a judge's reasoning. Langfuse stores it as the score's comment.

In [ ]:
case = "personal-review-date"
trace_id = baseline_scores.loc[case, "trace_id"]
for s in tracing.langfuse.api.scores_v3.get_many_v3(trace_id=trace_id, fields="details", limit=20).data:
    print(f"{s.name:18} {s.value!s:6} {s.comment}")

**Look in Langfuse:** **Experiments** → the baseline run. Each row is one test case with its scores; open one
to see its trace. Judge executions have their own traces in the `langfuse-llm-as-a-judge` environment.

What the run cost, as Langfuse priced it:

In [ ]:
experiment.experiment_costs("baseline")

## Part D: Your own evaluator

An evaluator is a plain function: it receives the item's `input`, the assistant's `output`, the
`expected_output` and `metadata`, and returns an `Evaluation`. This one costs nothing to run.

**Exercise:** finish `points_to_help`. It should give 1 when an answer that found no policy tells the person
where to go next (`hr-help@northwind.example` or Workday), and 0 when it does not. Answers that did find a
policy (`found_policy` is true) should get 1. Then run it on five test cases.

In [ ]:
from langfuse import Evaluation

def points_to_help(*, input, output, expected_output, metadata, **kwargs):
    text = output["answer"].lower()
    if output["found_policy"]:
        return Evaluation(name="points_to_help", value=1, comment="answered from policy")
    # TODO: check whether `text` mentions hr-help@northwind.example or Workday
    helped = ...
    return Evaluation(name="points_to_help", value=1 if helped else 0,
                      comment="points to help" if helped else "dead end")

In [ ]:
# The five "no answer in the documents" and "personal question" cases
picked = [i for i in items if i.metadata["trap"] in ("not in corpus", "personal question")][:5]
mine = tracing.langfuse.run_experiment(
    name="my-evaluator", data=picked, task=experiment.task,
    evaluators=[experiment.golden_check, points_to_help], max_concurrency=4)
tracing.flush()
pd.DataFrame([{"case": r.item.id, **{e.name: e.value for e in r.evaluations}} for r in mine.item_results])

Look at `small-talk`: "hello" gets a greeting, found no policy, and does not point anywhere, so it scores 0.
Is that a fault in the assistant or in the evaluator? Deciding what an evaluator should count is most of the
work of writing one. Change it so greetings pass, and run it again.

## Part E: Grade answers yourself

LLM judges are useful only if they agree with people. Put eight answers in a review queue, the ones a judge
or a check marked down first:

In [ ]:
queue_id, queued = experiment.create_review_queue("baseline")
print("queue:", queue_id)
print("answers to review:", queued)

**In Langfuse:** open **Human Annotation** → `Northwind review (baseline)`. For each answer, read the question,
the answer and its sources, then set `human_correctness` to *correct*, *partially correct* or *incorrect*, and
click **Complete + next**.

You can also score any single trace without a queue: open it in **Tracing** and use **Annotate**.

When you have graded a few, compare your verdicts with the judges':

In [ ]:
now = pd.DataFrame.from_dict(experiment.experiment_scores("baseline", wait_for_judges=False), orient="index")
reviewed = now[now.index.isin(queued)]
if "human_correctness" not in reviewed:
    print("No human_correctness scores yet: grade some answers in Human Annotation, then run this cell again.")
else:
    display(reviewed[["human_correctness", "correctness", "faithfulness", "golden_check"]])

Where you and a judge disagree, read the judge's reasoning. Is the judge too strict, too lenient, or did it
see something you missed? That decides whether to change its prompt.

## Part F: Catch a regression

"Does Northwind offer pet insurance?" scores 0.762 against the documents, so it reaches Claude. Someone
proposes raising the retrieval cut-off from 0.595 to 0.77 to stop that. Try it, for this run only:

In [ ]:
regression = experiment.run("regression-cutoff-0.77", judge=True, min_similarity=0.77)
experiment.summarise(regression)
print("cut-off back to:", config.MIN_BEST_SIMILARITY)

Compare the two runs, case by case:

In [ ]:
after = pd.DataFrame.from_dict(experiment.experiment_scores("regression-cutoff-0.77"), orient="index")
compare = pd.DataFrame({
    "trap": [i.metadata["trap"] for i in items],
    "check before": baseline_scores["golden_check"], "check after": after["golden_check"],
    "judge before": baseline_scores["correctness"], "judge after": after["correctness"],
}, index=[i.id for i in items])
print("average before:", compare[["check before", "judge before"]].mean().round(2).to_dict())
print("average after: ", compare[["check after", "judge after"]].mean().round(2).to_dict())
compare[compare["check after"] < compare["check before"]]

The pet-insurance question was never failing, while real questions now get "I could not find anything".
In CI the same comparison becomes a gate that stops the change:

In [ ]:
before, after_rate = compare["check before"].mean(), compare["check after"].mean()
if after_rate < before:
    print(f"BLOCKED: pass rate fell from {before:.0%} to {after_rate:.0%}")
else:
    print("OK to merge")

**Look in Langfuse:** **Experiments**, select both runs and compare them side by side.

What the regression run cost:

In [ ]:
experiment.experiment_costs("regression-cutoff-0.77")

## Wrap-up

- **Checks** (`golden_check`) are free and exact, but only see words.
- **LLM judges** read for meaning, and cost a fraction of a cent each with a small model.
- **People** decide whether to trust the judges.
- **Experiments** on a fixed dataset turn "it seems fine" into numbers you can compare.

The 👍 and 👎 under each answer in the chat page (http://localhost:8501) record a `user_feedback` score on
that answer's trace. Try one, then find it in **Scores**.